# Paper 02 — Phase 03: Mechanism-Gate Experiment (450-Run Grid)

**Project:** *Critical Compositional Pressure & The Two-Subspace Law*  
**Hardware Target:** Kaggle (2x Tesla T4) or Google Colab (T4 / A100)  
**Framework:** Research Planning Framework (RPF v2.0.0 — P03 Mechanism Gate)  

---

## Overview & Protocol Structure
This notebook executes the full **450-run empirical mechanism gate**:
- **Arm A (Dense $\lambda$-Sweep, 300 runs):** $\lambda \in \{0.00, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00, 1.50, 2.00\} \times 30\text{ seeds}$
- **Arm B (Late-Onset Intervention, 150 runs):** $t_{\text{int}} \in \{0, 100, 250, 500, 1000\text{ steps}\} \times 30\text{ seeds}$ with $\lambda = 1.0$
- **Architecture:** 2-layer seq2seq Transformer ($d_{\text{model}} = 128, n_{\text{heads}} = 4, d_{\text{ff}} = 512, n_{\text{layers}} = 2$)
- **Benchmark:** H-Bar zero-leakage compositional suite ($N_{\text{train}} = 10,000, N_{\text{OOD}} = 2,000$)
- **Seed Pinning:** Strictly $\text{seed} = \text{run\_id} \times 42 + 7$

## Expected Runtime on Kaggle (T4 GPU)
- ~20–25 seconds per 2000-step run with PyTorch AMP.
- **Total Runtime (450 runs):** ~2.5 to 3.0 hours (well within Kaggle's 9-hour session limit).

## Generated Artifacts (in `/kaggle/working/gate_output/`)
1. `all_results.pkl` — Consolidated dictionary containing metrics across all 450 runs.
2. `gate_separatrix_results.png` — Publication-grade 3-panel figure.
3. `gate_summary_table.csv` — Full per-condition statistical summary table.
4. `gate_output.zip` — Compressed bundle containing all outputs for immediate download.

In [ ]:
# =============================================================================
# CELL 1: Environment Setup & Hardware Acceleration Check
# =============================================================================
import os
import sys
import math
import time
import random
import pickle
import zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Any, Tuple, List, Dict, Set

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import t as student_t
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler

# Determinism flags (CC.1.2 / AGENTS.md reproducibility convention)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -----------------------------------------------------------------------------
# Grid configuration - single source of truth (mirror of
# paper02/experiments/configs/gate_protocol.yaml). Reused by Cells 6-9.
# -----------------------------------------------------------------------------
NUM_SEEDS = 30
TOTAL_STEPS = 2000
EVAL_INTERVAL = 25
LAMBDA_SWEEP = [0.00, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00, 1.50, 2.00]
INTERVENTION_STEPS = [0, 100, 250, 500, 1000]
TOTAL_GATE_RUNS = (len(LAMBDA_SWEEP) + len(INTERVENTION_STEPS)) * NUM_SEEDS

# Directory setup
BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else "."
OUT_DIR = Path(BASE_DIR) / "gate_output"
RUNS_DIR = OUT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print("=" * 75)
print(f"PyTorch Version : {torch.__version__}")
print(f"Compute Device  : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"AMP Acceleration: {USE_AMP}")
print(f"Output Directory: {OUT_DIR}")
if not torch.cuda.is_available():
    print("\n⚠️ WARNING: CUDA GPU is NOT detected! You are running on CPU.")
    print(f"   On CPU, {TOTAL_GATE_RUNS} runs take ~30 hours! To fix this in Kaggle:")
    print("   👉 Click 'Settings' (right panel) -> 'Accelerator' -> Select 'GPU T4 x 2' or 'GPU T4'.\n")
else:
    print("\n✅ GPU Acceleration is ACTIVE! Estimated runtime: ~2.5 - 3.0 hours.\n")
print("=" * 75)

In [ ]:
# =============================================================================
# CELL 2: Synthetic Benchmark Generator (H-Bar Zero-Leakage Suite)
# =============================================================================
PRIMITIVES = {
    "jump": "JUMP", "walk": "WALK", "run": "RUN", "look": "LOOK",
    "left": "LTURN", "right": "RTURN", "twice": "X2", "thrice": "X3",
    "and": "AND", "after": "AFTER", "around": "AROUND", "opposite": "OPPOSITE",
}

def gen_simple(rng: random.Random) -> Tuple[str, str]:
    base = rng.choice(["walk", "run", "jump", "look"])
    r = rng.random()
    if r > 0.7:
        d = rng.choice(["left", "right"])
        return (f"{base} {d}", f"{PRIMITIVES[base]} {PRIMITIVES[d]}")
    elif r > 0.4:
        m = rng.choice(["twice", "thrice"])
        return (f"{m} {base}", f"{PRIMITIVES[m]} {PRIMITIVES[base]}")
    else:
        return (base, PRIMITIVES[base])

def gen_id_medium(rng: random.Random) -> Tuple[str, str]:
    base = rng.choice(["walk", "run", "jump", "look"])
    parts, actions = [base], [PRIMITIVES[base]]
    if rng.random() > 0.5:
        d = rng.choice(["left", "right"])
        parts.append(d); actions.append(PRIMITIVES[d])
    if rng.random() > 0.6:
        m = rng.choice(["twice", "thrice"])
        parts.insert(0, m); actions.insert(0, PRIMITIVES[m])
    return (" ".join(parts), " ".join(actions))

def gen_hard_ood(rng: random.Random) -> Tuple[str, str]:
    r = rng.random()
    if r < 0.25:
        mod = rng.choice(["twice", "thrice"]); d = rng.choice(["left", "right"])
        return (f"{mod} jump {d}", f"{PRIMITIVES[mod]} JUMP {PRIMITIVES[d]}")
    elif r < 0.45:
        base = rng.choice(["jump", "walk"]); d = rng.choice(["left", "right"])
        return (f"opposite {base} {d}", f"OPPOSITE {PRIMITIVES[base]} {PRIMITIVES[d]}")
    elif r < 0.65:
        b1 = rng.choice(["jump", "walk"]); b2 = rng.choice(["run", "look"]); conj = rng.choice(["and", "after"])
        if conj == "and":
            return (f"{b1} {conj} {b2}", f"{PRIMITIVES[b1]} {PRIMITIVES[conj]} {PRIMITIVES[b2]}")
        return (f"{conj} {b1} {b2}", f"{PRIMITIVES[conj]} {PRIMITIVES[b1]} {PRIMITIVES[b2]}")
    elif r < 0.80:
        d = rng.choice(["left", "right"])
        return (f"jump around {d} twice", f"JUMP AROUND {PRIMITIVES[d]} X2")
    mods = rng.sample(["twice", "thrice", "opposite"], 2)
    d = rng.choice(["left", "right"])
    return (f"{mods[0]} {mods[1]} jump {d}", f"{PRIMITIVES[mods[0]]} {PRIMITIVES[mods[1]]} JUMP {PRIMITIVES[d]}")

@dataclass(frozen=True)
class HBarDataSplits:
    train_pairs: List[Tuple[str, str]]
    id_pairs: List[Tuple[str, str]]
    ood_pairs: List[Tuple[str, str]]
    comp_pairs: List[Tuple[str, str]]
    vocab: Dict[str, int]
    inv_vocab: Dict[int, str]

def generate_hbar_splits(n_train: int = 10000, n_test_id: int = 2000, n_test_ood: int = 2000, n_comp_probe: int = 2000, seed: int = 42) -> HBarDataSplits:
    rng = random.Random(seed)
    train_pairs = [gen_simple(rng) for _ in range(n_train)]
    id_pairs = [gen_id_medium(rng) for _ in range(n_test_id)]
    ood_pairs = [gen_hard_ood(rng) for _ in range(n_test_ood)]
    comp_pairs = [gen_hard_ood(rng) for _ in range(n_comp_probe)]

    train_cmds = {c for c, _ in train_pairs}
    ood_cmds = {c for c, _ in ood_pairs}
    assert len(train_cmds.intersection(ood_cmds)) == 0, "Zero-leakage invariant violated!"

    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
    for pairs in [train_pairs, id_pairs, ood_pairs, comp_pairs]:
        for c, a in pairs:
            for tok in c.split() + a.split():
                if tok not in vocab:
                    vocab[tok] = len(vocab)
    inv_vocab = {v: k for k, v in vocab.items()}
    return HBarDataSplits(train_pairs, id_pairs, ood_pairs, comp_pairs, vocab, inv_vocab)

class HBarDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]], vocab: Dict[str, int]):
        self.pairs = pairs
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        cmd, act = self.pairs[idx]
        src_tokens = [self.vocab[tok] for tok in cmd.split()]
        act_tokens = [self.vocab[tok] for tok in act.split()]
        tgt_in = [self.vocab["<SOS>"]] + act_tokens
        tgt_out = act_tokens + [self.vocab["<EOS>"]]
        return {
            "src": torch.tensor(src_tokens, dtype=torch.long),
            "tgt_in": torch.tensor(tgt_in, dtype=torch.long),
            "tgt_out": torch.tensor(tgt_out, dtype=torch.long),
        }

def collate_hbar_batch(batch: List[Dict[str, torch.Tensor]], pad_idx: int = 0) -> Dict[str, torch.Tensor]:
    src_padded = torch.nn.utils.rnn.pad_sequence([item["src"] for item in batch], batch_first=True, padding_value=pad_idx)
    tgt_in_padded = torch.nn.utils.rnn.pad_sequence([item["tgt_in"] for item in batch], batch_first=True, padding_value=pad_idx)
    tgt_out_padded = torch.nn.utils.rnn.pad_sequence([item["tgt_out"] for item in batch], batch_first=True, padding_value=pad_idx)
    return {"src": src_padded, "tgt_in": tgt_in_padded, "tgt_out": tgt_out_padded}

def create_hbar_dataloaders(splits: HBarDataSplits, batch_size: int = 64, seed: int = 42, num_workers: int = 0) -> Dict[str, DataLoader]:
    def _init_fn(worker_id):
        np.random.seed(seed + worker_id)
        random.seed(seed + worker_id)
    kw_tr = dict(batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=len(splits.train_pairs)>=batch_size, collate_fn=lambda b: collate_hbar_batch(b, pad_idx=splits.vocab["<PAD>"]), worker_init_fn=_init_fn)
    kw_comp = dict(batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=len(splits.comp_pairs)>=batch_size, collate_fn=lambda b: collate_hbar_batch(b, pad_idx=splits.vocab["<PAD>"]), worker_init_fn=_init_fn)
    kw_ev = dict(batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=lambda b: collate_hbar_batch(b, pad_idx=splits.vocab["<PAD>"]), worker_init_fn=_init_fn)
    return {
        "train": DataLoader(HBarDataset(splits.train_pairs, splits.vocab), **kw_tr),
        "id": DataLoader(HBarDataset(splits.id_pairs, splits.vocab), **kw_ev),
        "ood": DataLoader(HBarDataset(splits.ood_pairs, splits.vocab), **kw_ev),
        "comp": DataLoader(HBarDataset(splits.comp_pairs, splits.vocab), **kw_comp),
    }

print("Generating canonical H-Bar dataset splits (N_train=10,000, N_ood=2,000)...")
DATA_SPLITS = generate_hbar_splits(10000, 2000, 2000, 2000, seed=42)
print(f"H-Bar benchmark ready! Vocab size = {len(DATA_SPLITS.vocab)}")

In [ ]:
# =============================================================================
# CELL 3: 2-Layer Seq2Seq Transformer Model & Representation Hook
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x + self.pe[:, : x.size(1), :])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, nhead: int = 4, num_layers: int = 2, dim_ff: int = 512, dropout: float = 0.1, pad_idx: int = 0):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_ff, dropout=dropout, batch_first=True,
        )
        self.output_proj = nn.Linear(d_model, vocab_size)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src: torch.Tensor) -> torch.Tensor:
        src_emb = self.pos_encoding(self.embedding(src) * math.sqrt(self.d_model))
        src_pad_mask = (src == self.pad_idx)
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_pad_mask)

    def encode_pooled(self, src: torch.Tensor) -> torch.Tensor:
        memory = self.encode(src)
        mask = (src != self.pad_idx).unsqueeze(-1).float()
        sum_pooled = (memory * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1.0)
        return sum_pooled / denom

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        memory = self.encode(src)
        src_pad_mask = (src == self.pad_idx)
        tgt_pad_mask = (tgt == self.pad_idx)
        tgt_emb = self.pos_encoding(self.embedding(tgt) * math.sqrt(self.d_model))
        tgt_len = tgt.size(1)
        tgt_mask = torch.triu(torch.ones(tgt_len, tgt_len, device=tgt.device), diagonal=1).bool()
        out = self.transformer.decoder(
            tgt_emb, memory, tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask, memory_key_padding_mask=src_pad_mask,
        )
        return self.output_proj(out)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

test_model = Seq2SeqTransformer(vocab_size=len(DATA_SPLITS.vocab))
print(f"Transformer instantiated successfully! Total trainable parameters: {test_model.count_parameters():,}")

In [ ]:
# =============================================================================
# CELL 4: Representational Geometry Alignment (RGA) Metric
# =============================================================================
RGA_OPERATOR_TOKENS = ("X2", "X3", "OPPOSITE", "AFTER", "AND", "AROUND")

def compute_rga_metric(model: nn.Module, sample_pairs: List[Tuple[str, str]], vocab: Dict[str, int], device: torch.device, operators: Tuple[str, ...] = RGA_OPERATOR_TOKENS, n_random_pairs: int = 2000, seed: int = 42) -> float:
    if len(sample_pairs) < 4:
        return 0.5
    op_ids = {vocab[op] for op in operators if op in vocab}
    was_training = model.training
    model.eval()
    try:
        pad_idx = vocab.get("<PAD>", 0)
        src_tensors = [torch.tensor([vocab[t] for t in cmd.split() if t in vocab], dtype=torch.long) for cmd, _ in sample_pairs]
        src_batch = torch.nn.utils.rnn.pad_sequence(src_tensors, batch_first=True, padding_value=pad_idx).to(device)

        with torch.inference_mode():
            with autocast("cuda", enabled=(device.type == "cuda")):
                pooled = model.encode_pooled(src_batch).float()
            pooled = F.normalize(pooled, p=2, dim=1)
            sim_matrix = torch.matmul(pooled, pooled.t())

        n = len(sample_pairs)
        labels = [{vocab[t] for t in act.split() if t in vocab}.intersection(op_ids) for _, act in sample_pairs]
        same_pairs = [(i, j) for i in range(n) for j in range(i + 1, n) if labels[i].intersection(labels[j])]
        if not same_pairs:
            return 0.0

        i_idx = torch.tensor([i for i, _ in same_pairs], dtype=torch.long, device=device)
        j_idx = torch.tensor([j for _, j in same_pairs], dtype=torch.long, device=device)
        mean_same = sim_matrix[i_idx, j_idx].mean().item()

        rng = random.Random(seed)
        rand_pairs = [(rng.randrange(n), rng.randrange(n)) for _ in range(min(n_random_pairs, n * (n - 1) // 2))]
        r_i = torch.tensor([i for i, _ in rand_pairs], dtype=torch.long, device=device)
        r_j = torch.tensor([j for _, j in rand_pairs], dtype=torch.long, device=device)
        mean_random = sim_matrix[r_i, r_j].mean().item()
        return float(min(1.0, max(0.0, mean_same - mean_random)))
    finally:
        model.train(was_training)

print("RGA metric tracking ready.")

In [ ]:
# =============================================================================
# CELL 5: Gate Execution Engine & Evaluator
# =============================================================================
@dataclass
class GateRunConfig:
    condition_type: str
    condition_name: str
    lambda_val: float
    intervention_step: int = None
    run_id: int = 0
    total_steps: int = 2000
    eval_interval: int = 25
    eval_max_batches: int = 8
    lr: float = 1e-3
    batch_size: int = 64
    d_model: int = 128
    nhead: int = 4
    num_layers: int = 2
    dim_ff: int = 512
    dropout: float = 0.1
    gradient_clip_norm: float = 1.0

def evaluate_loader_accuracy(model: nn.Module, loader: DataLoader, device: torch.device, use_amp: bool = False, max_batches: int = None) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            src = batch["src"].to(device)
            tgt_in = batch["tgt_in"].to(device)
            tgt_out = batch["tgt_out"].to(device)
            with autocast("cuda", enabled=(use_amp and device.type == "cuda")):
                logits = model(src, tgt_in)
            preds = logits.argmax(dim=-1)
            mask = (tgt_out != 0)
            correct += ((preds == tgt_out) & mask).sum().item()
            total += mask.sum().item()
    model.train()
    return float(100.0 * correct / total) if total > 0 else 0.0

def train_gate_run(config: GateRunConfig, data_splits: HBarDataSplits, device: torch.device) -> Dict[str, Any]:
    seed = config.run_id * 42 + 7
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    vocab_size = len(data_splits.vocab)
    model = Seq2SeqTransformer(
        vocab_size=vocab_size, d_model=config.d_model, nhead=config.nhead,
        num_layers=config.num_layers, dim_ff=config.dim_ff, dropout=config.dropout,
    ).to(device)

    eff_batch_size = min(config.batch_size, len(data_splits.train_pairs), len(data_splits.comp_pairs))
    loaders = create_hbar_dataloaders(splits=data_splits, batch_size=eff_batch_size, seed=seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, betas=(0.9, 0.999), eps=1e-8)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scaler = GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))
    comp_probe_sample = data_splits.comp_pairs[:128]

    metrics = {"step": [], "loss_train": [], "loss_comp": [], "acc_id": [], "acc_ood": [], "cka_rga": [], "param_norm": []}
    train_iter = iter(loaders["train"])
    comp_iter = iter(loaders["comp"])

    for step in range(config.total_steps):
        if config.condition_type == "arm_a_lambda":
            eff_lambda = config.lambda_val
        elif config.condition_type == "arm_b_late_intervention":
            t_int = config.intervention_step or 0
            eff_lambda = config.lambda_val if step >= t_int else 0.0
        else:
            eff_lambda = config.lambda_val

        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(loaders["train"]); batch = next(train_iter)

        src = batch["src"].to(device); tgt_in = batch["tgt_in"].to(device); tgt_out = batch["tgt_out"].to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=(USE_AMP and device.type == "cuda")):
            logits = model(src, tgt_in)
            loss_train = criterion(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
            loss_comp_val = 0.0
            if eff_lambda > 0.0:
                try:
                    c_batch = next(comp_iter)
                except StopIteration:
                    comp_iter = iter(loaders["comp"]); c_batch = next(comp_iter)
                c_src = c_batch["src"].to(device); c_tgt_in = c_batch["tgt_in"].to(device); c_tgt_out = c_batch["tgt_out"].to(device)
                c_logits = model(c_src, c_tgt_in)
                loss_comp = criterion(c_logits.reshape(-1, vocab_size), c_tgt_out.reshape(-1))
                loss_comp_val = float(loss_comp.item())
                total_loss = loss_train + eff_lambda * loss_comp
            else:
                total_loss = loss_train

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        if step % config.eval_interval == 0 or step == config.total_steps - 1:
            acc_id = evaluate_loader_accuracy(model, loaders["id"], device, USE_AMP, max_batches=config.eval_max_batches)
            acc_ood = evaluate_loader_accuracy(model, loaders["ood"], device, USE_AMP, max_batches=config.eval_max_batches)
            rga_val = compute_rga_metric(model, comp_probe_sample, data_splits.vocab, device, seed=seed)
            param_norm = float(sum(p.detach().pow(2).sum().item() for p in model.parameters()) ** 0.5)

            metrics["step"].append(step)
            metrics["loss_train"].append(float(loss_train.item()))
            metrics["loss_comp"].append(loss_comp_val)
            metrics["acc_id"].append(acc_id)
            metrics["acc_ood"].append(acc_ood)
            metrics["cka_rga"].append(rga_val)
            metrics["param_norm"].append(param_norm)

    final_acc_id = evaluate_loader_accuracy(model, loaders["id"], device, USE_AMP, max_batches=None)
    final_acc_ood = evaluate_loader_accuracy(model, loaders["ood"], device, USE_AMP, max_batches=None)
    res = {
        "config": asdict(config),
        "seed": seed,
        "metrics": metrics,
        "final": {
            "acc_id": final_acc_id,
            "acc_ood": final_acc_ood,
            "escaped": final_acc_ood >= 80.0,
            "final_param_norm": metrics["param_norm"][-1],
            "final_rga": metrics["cka_rga"][-1],
        }
    }
    del model, optimizer
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return res

In [ ]:
# =============================================================================
# CELL 6: Build Full Gate Experimental Matrix Configuration
# =============================================================================
# Grid constants (NUM_SEEDS, TOTAL_STEPS, EVAL_INTERVAL, LAMBDA_SWEEP,
# INTERVENTION_STEPS) are defined in Cell 1 - single source of truth.

all_configs = []

# Arm A: Dense Lambda Sweep (computed below)
for l_val in LAMBDA_SWEEP:
    for s in range(NUM_SEEDS):
        all_configs.append(GateRunConfig(
            condition_type="arm_a_lambda",
            condition_name=f"arm_a_lambda_{l_val:.2f}",
            lambda_val=l_val,
            run_id=s,
            total_steps=TOTAL_STEPS,
            eval_interval=EVAL_INTERVAL,
        ))

# Arm B: Late-Onset Intervention (computed below)
for t_int in INTERVENTION_STEPS:
    for s in range(NUM_SEEDS):
        all_configs.append(GateRunConfig(
            condition_type="arm_b_late_intervention",
            condition_name=f"arm_b_tint_{t_int}",
            lambda_val=1.00,
            intervention_step=t_int,
            run_id=s,
            total_steps=TOTAL_STEPS,
            eval_interval=EVAL_INTERVAL,
        ))

print("=" * 75)
print(f"Mechanism Gate Suite Configured: Total Planned Runs = {len(all_configs)}")
print(f"Arm A (Dense Lambda Sweep)  : {len(LAMBDA_SWEEP)} levels x {NUM_SEEDS} seeds = {len(LAMBDA_SWEEP) * NUM_SEEDS} runs")
print(f"Arm B (Late Intervention)   : {len(INTERVENTION_STEPS)} levels x {NUM_SEEDS} seeds = {len(INTERVENTION_STEPS) * NUM_SEEDS} runs")
print("=" * 75)

In [ ]:
# =============================================================================
# CELL 7: Execute Full Gate Run Matrix with Real-Time Per-Run Progress Tracking
# =============================================================================
start_time = time.time()
all_results = []

print(f"\n🚀 Starting High-Throughput Mechanism Gate Execution ({len(all_configs)} runs)...\n", flush=True)
grid_pbar = tqdm(all_configs, desc="Overall Gate Progress", unit="run")

for idx, run_cfg in enumerate(grid_pbar, 1):
    t0 = time.time()
    print(f"[{idx:03d}/{len(all_configs)}] Running {run_cfg.condition_name} (seed {run_cfg.run_id:02d})... ", end="", flush=True)
    
    res = train_gate_run(run_cfg, DATA_SPLITS, device=DEVICE)
    elapsed = time.time() - t0
    all_results.append(res)

    # Save per-run pickle
    run_file = RUNS_DIR / f"{run_cfg.condition_name}_run{run_cfg.run_id:02d}.pkl"
    with open(run_file, "wb") as f:
        pickle.dump(res, f)

    total_elapsed = time.time() - start_time
    avg_per_run = total_elapsed / idx
    eta_hrs = (avg_per_run * (len(all_configs) - idx)) / 3600.0
    
    print(f"Done ({elapsed:.1f}s) | OOD Acc: {res['final']['acc_ood']:.1f}% | Escaped: {res['final']['escaped']} | ETA: {eta_hrs:.2f}h", flush=True)
    grid_pbar.set_postfix({"Last OOD": f"{res['final']['acc_ood']:.1f}%", "ETA": f"{eta_hrs:.2f}h"})

# Save consolidated results dictionary
all_results_path = OUT_DIR / "all_results.pkl"
with open(all_results_path, "wb") as f:
    pickle.dump({
        "timestamp": datetime.now().isoformat(),
        "total_runs": len(all_results),
        "runs": all_results,
    }, f)

print("\n" + "=" * 75)
print(f"All {len(all_results)} Runs Complete in {(time.time() - start_time)/3600.0:.2f} hours!")
print(f"Consolidated file written to: {all_results_path}")
print("=" * 75)

In [ ]:
# =============================================================================
# CELL 8: Pre-Registered Statistical Analysis & Formal Gate Verdict
# =============================================================================
def logistic_step_fn(lambda_val: np.ndarray, k: float, lambda_crit: float) -> np.ndarray:
    z = -k * (lambda_val - lambda_crit)
    return 1.0 / (1.0 + np.exp(np.clip(z, -50.0, 50.0)))

def fit_logistic_separatrix(lambda_vals: np.ndarray, escape_fractions: np.ndarray) -> Tuple[float, float, float]:
    # Guard: need >= 4 cells to fit a 2-parameter change-point model (mirrors analyze_gate.py).
    if len(lambda_vals) < 4:
        return 0.0, 0.0, 0.0
    try:
        p0 = [20.0, float(np.median(lambda_vals))]
        bounds = ([0.1, 0.0], [200.0, float(np.max(lambda_vals))])
        popt, _ = curve_fit(logistic_step_fn, lambda_vals, escape_fractions, p0=p0, bounds=bounds, maxfev=5000)
        k_fit, lam_crit_fit = float(popt[0]), float(popt[1])
        y_pred = logistic_step_fn(lambda_vals, k_fit, lam_crit_fit)
        ss_res = float(np.sum((escape_fractions - y_pred) ** 2))
        ss_tot = float(np.sum((escape_fractions - np.mean(escape_fractions)) ** 2))
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 1e-12 else 1.0
        return k_fit, lam_crit_fit, max(0.0, r2)
    except Exception:
        return 0.0, 0.0, 0.0

def compute_bootstrap_ci(data: np.ndarray, num_bootstrap: int = 2000, ci: float = 0.95, seed: int = 42) -> Tuple[float, float]:
    # Guard: empty cell -> degenerate CI (mirrors analyze_gate.py).
    if len(data) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    boot_means = np.array([np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(num_bootstrap)])
    alpha = (1.0 - ci) / 2.0
    return float(np.percentile(boot_means, 100.0 * alpha)), float(np.percentile(boot_means, 100.0 * (1.0 - alpha)))

def compute_tost_equivalence(sample_a: np.ndarray, sample_b: np.ndarray, delta: float = 2.5, alpha: float = 0.05) -> Tuple[bool, float, float]:
    n_a, n_b = len(sample_a), len(sample_b)
    mean_a, mean_b = np.mean(sample_a), np.mean(sample_b)
    var_a, var_b = np.var(sample_a, ddof=1), np.var(sample_b, ddof=1)
    diff = mean_a - mean_b
    se = np.sqrt(var_a / n_a + var_b / n_b)
    if se < 1e-12:
        return True, 0.0, 0.0
    dof = (var_a / n_a + var_b / n_b) ** 2 / ((var_a / n_a) ** 2 / (n_a - 1) + (var_b / n_b) ** 2 / (n_b - 1))
    t_lower = (diff + delta) / se
    p_lower = 1.0 - float(student_t.cdf(t_lower, df=dof))
    t_upper = (delta - diff) / se
    p_upper = 1.0 - float(student_t.cdf(t_upper, df=dof))
    return bool(max(p_lower, p_upper) < alpha), float(p_lower), float(p_upper)

# -----------------------------------------------------------------------------
# Evaluate Pre-Registered Criteria
# -----------------------------------------------------------------------------
arm_a_runs = [r for r in all_results if r["config"]["condition_type"] == "arm_a_lambda"]
arm_b_runs = [r for r in all_results if r["config"]["condition_type"] == "arm_b_late_intervention"]

arm_a_by_lambda = {}
for r in arm_a_runs:
    arm_a_by_lambda.setdefault(float(r["config"]["lambda_val"]), []).append(r)
sorted_lams = sorted(arm_a_by_lambda.keys())

# Criterion 1: Separatrix Step Sharpness
escape_fractions = [sum(1 for r in arm_a_by_lambda[l] if r["final"]["acc_ood"] >= 80.0) / len(arm_a_by_lambda[l]) for l in sorted_lams]
k_fit, lam_crit_fit, r2_fit = fit_logistic_separatrix(np.array(sorted_lams), np.array(escape_fractions))
sub_cells_frac = [escape_fractions[i] for i, l in enumerate(sorted_lams) if l <= 0.10]
super_cells_frac = [escape_fractions[i] for i, l in enumerate(sorted_lams) if l >= 0.50]
sub_esc = max(sub_cells_frac) if sub_cells_frac else 0.0
super_esc = min(super_cells_frac) if super_cells_frac else 1.0
pass_crit_1 = (k_fit >= 15.0 and sub_esc < 0.05 and super_esc > 0.95)

# Criterion 2: Late-Onset Destabilization at t_int = 1000
# Pre-registered population: t_int = 1000 runs TRAPPED at step 1000
# (Acc_OOD <= 50% at step 1000; preregistration.md section 2 / P03_GATE.md Task 3.3).
def _acc_ood_at_step(run: Dict[str, Any], target_step: int) -> float | None:
    steps = run["metrics"]["step"]
    if target_step in steps:
        return float(run["metrics"]["acc_ood"][steps.index(target_step)])
    below = [s for s in steps if s <= target_step]
    if not below:
        return None
    return float(run["metrics"]["acc_ood"][steps.index(max(below))])

def _is_trapped_at_1000(run: Dict[str, Any]) -> bool:
    acc_at_1000 = _acc_ood_at_step(run, 1000)
    return acc_at_1000 is not None and acc_at_1000 <= 50.0

arm_b_t1000 = [r for r in arm_b_runs if r["config"].get("intervention_step") == 1000]
trapped_t1000 = [r for r in arm_b_t1000 if _is_trapped_at_1000(r)]
n_trapped = len(trapped_t1000)
recovered_t1000 = sum(1 for r in trapped_t1000 if r["final"]["acc_ood"] >= 90.0)
rec_frac = recovered_t1000 / n_trapped if n_trapped > 0 else 0.0
pass_crit_2 = (n_trapped > 0 and rec_frac >= 0.90)

# Criterion 3: Supercritical TOST Equivalence
super_target = [0.50, 0.75, 1.00, 1.50, 2.00]
present_super = [l_val for l_val in super_target if l_val in arm_a_by_lambda]
tost_pass = True
num_pairs = len(present_super) * (len(present_super) - 1) // 2
bonf_alpha = 0.05 / max(1, num_pairs)
for i in range(len(present_super)):
    for j in range(i + 1, len(present_super)):
        s1 = np.array([r["final"]["acc_ood"] for r in arm_a_by_lambda[present_super[i]]])
        s2 = np.array([r["final"]["acc_ood"] for r in arm_a_by_lambda[present_super[j]]])
        equiv, _, _ = compute_tost_equivalence(s1, s2, delta=2.5, alpha=bonf_alpha)
        if not equiv:
            tost_pass = False
pass_crit_3 = (tost_pass and len(present_super) >= 2)

# -----------------------------------------------------------------------------
# Pre-Registered Decision Rule (preregistration.md section 5 decision_rule)
#   PASS         = primary (C1) meets threshold AND secondaries (C2, C3) consistent
#   FAIL         = primary unmet (or a secondary inconsistent)
#   INCONCLUSIVE = k in [10, 15) AND >= 1 separation inequality violated by < 2 SE
#                  -> triggers the P03.1 sub-gate protocol (planning/phases/P03.1_subgate.md)
# -----------------------------------------------------------------------------
def _escape_fraction_se(p: float, n: int) -> float:
    return float(math.sqrt(p * (1.0 - p) / n)) if n > 0 else 0.0

max_sep_violation, sep_violation_se = 0.0, 0.0
if sub_esc >= 0.05:
    sub_cells = [(l_val, escape_fractions[i]) for i, l_val in enumerate(sorted_lams) if l_val <= 0.10]
    worst_l, worst_p = max(sub_cells, key=lambda t: t[1])
    violation = worst_p - 0.05
    if violation > max_sep_violation:
        max_sep_violation, sep_violation_se = violation, _escape_fraction_se(worst_p, len(arm_a_by_lambda[worst_l]))
if super_esc <= 0.95:
    super_cells = [(l_val, escape_fractions[i]) for i, l_val in enumerate(sorted_lams) if l_val >= 0.50]
    worst_l, worst_p = min(super_cells, key=lambda t: t[1])
    violation = 0.95 - worst_p
    if violation > max_sep_violation:
        max_sep_violation, sep_violation_se = violation, _escape_fraction_se(worst_p, len(arm_a_by_lambda[worst_l]))

in_ambiguity_band = (
    10.0 <= k_fit < 15.0
    and max_sep_violation > 0.0
    and max_sep_violation < 2.0 * sep_violation_se
)

if pass_crit_1 and pass_crit_2 and pass_crit_3:
    verdict = "PASS"
elif in_ambiguity_band:
    verdict = "INCONCLUSIVE"
else:
    verdict = "FAIL"

print("=" * 75)
print(f"MECHANISM GATE VERDICT: {verdict}" + (" (triggering P03.1 sub-gate)" if verdict == "INCONCLUSIVE" else ""))
print("=" * 75)
print(f"Criterion 1 (Separatrix Step): {'PASS' if pass_crit_1 else 'FAIL'} | k = {k_fit:.2f} (req >= 15.0), lambda_crit = {lam_crit_fit:.3f}, sub = {sub_esc:.2f}, super = {super_esc:.2f}")
if n_trapped > 0:
    excluded_t1000 = len(arm_b_t1000) - n_trapped
    print(f"Criterion 2 (Late-Onset 1000): {'PASS' if pass_crit_2 else 'FAIL'} | Recovery = {rec_frac*100:.1f}% ({recovered_t1000}/{n_trapped} trapped seeds) (req >= 90%)" + (f" | {excluded_t1000} non-trapped t_int=1000 run(s) excluded per pre-registration" if excluded_t1000 else ""))
else:
    print("Criterion 2 (Late-Onset 1000): FAIL | not evaluable (no t_int=1000 runs trapped at step 1000)")
print(f"Criterion 3 (TOST Equiv +/-2.5%): {'PASS' if pass_crit_3 else 'FAIL'} | {num_pairs} supercritical pair(s) tested (Bonferroni alpha = {bonf_alpha:.4f})")
print("=" * 75)

In [ ]:
# =============================================================================
# CELL 9: Plot 3-Panel Diagnostic Figure & Package Outputs
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Separatrix Fit
ax1 = axes[0]
ax1.scatter(sorted_lams, escape_fractions, color="#1f77b4", s=70, zorder=3, label="Empirical Escape Fraction (n=30)")
dense_lams = np.linspace(0.0, 2.0, 200)
ax1.plot(dense_lams, logistic_step_fn(dense_lams, k_fit, lam_crit_fit), color="#d62728", lw=2.5, label=f"Logistic Fit ($k={k_fit:.1f}, \hat{{\lambda}}_{{\mathrm{{crit}}}}={lam_crit_fit:.2f}$)")
ax1.axvline(lam_crit_fit, color="gray", linestyle="--", alpha=0.7, label=r"Separatrix $\lambda_{\mathrm{crit}}$")
ax1.set_title(r"(A) Separatrix Escape Probability $P(\mathrm{escape} \mid \lambda)$", fontsize=12, fontweight="bold")
ax1.set_xlabel(r"Compositional Pressure $\lambda$", fontsize=11)
ax1.set_ylabel(r"Escape Probability $P(\mathrm{Acc}_{\mathrm{OOD}} \geq 80\%)$", fontsize=11)
ax1.set_ylim(-0.05, 1.05); ax1.legend(frameon=True, fontsize=9)

# Panel B: Late-Onset Trajectories
ax2 = axes[1]
tint_groups = {}
for r in arm_b_runs:
    tint_groups.setdefault(r["config"].get("intervention_step", 0) or 0, []).append(r)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(tint_groups)))
for (t_int, runs), col in zip(sorted(tint_groups.items()), colors):
    traj_mat = np.array([r["metrics"]["acc_ood"] for r in runs])
    steps = runs[0]["metrics"]["step"]
    ax2.plot(steps, np.mean(traj_mat, axis=0), color=col, lw=2.0, label=f"$t_{{\mathrm{{int}}}} = {t_int}$")
    ax2.axvline(t_int, color=col, linestyle=":", alpha=0.5)
ax2.set_title("(B) Late-Onset Destabilization & Recovery", fontsize=12, fontweight="bold")
ax2.set_xlabel("Training Steps", fontsize=11); ax2.set_ylabel("OOD Accuracy (%)", fontsize=11)
ax2.set_ylim(-5, 105); ax2.legend(frameon=True, fontsize=9)

# Panel C: Supercritical Distributions
ax3 = axes[2]
box_data = [[r["final"]["acc_ood"] for r in arm_a_by_lambda[l]] for l in sorted_lams]
bp = ax3.boxplot(box_data, tick_labels=[f"{l:.2f}" for l in sorted_lams], patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#aec7e8")
ax3.set_title(r"(C) OOD Accuracy Distributions across $\lambda$", fontsize=12, fontweight="bold")
ax3.set_xlabel(r"Compositional Pressure $\lambda$", fontsize=11); ax3.set_ylabel("Final OOD Accuracy (%)", fontsize=11)
ax3.set_ylim(-5, 105)

plt.tight_layout()
fig_path = OUT_DIR / "gate_separatrix_results.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Diagnostic figure saved to: {fig_path}")

# Summary Table CSV
summary_rows = []
grouped = {}
for r in all_results:
    grouped.setdefault(r["config"]["condition_name"], []).append(r)
for c_name, runs in grouped.items():
    oods = np.array([r["final"]["acc_ood"] for r in runs])
    low_ci, high_ci = compute_bootstrap_ci(oods)
    summary_rows.append({
        "Condition": c_name,
        "Mean_OOD": np.mean(oods),
        "Std_OOD": np.std(oods, ddof=1),
        "Median_OOD": np.median(oods),
        "CI_95_Low": low_ci,
        "CI_95_High": high_ci,
        "Escape_Fraction": sum(1 for r in runs if r["final"]["acc_ood"] >= 80.0) / len(runs),
    })
summary_df = pd.DataFrame(summary_rows)
summary_csv = OUT_DIR / "gate_summary_table.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Summary CSV saved to: {summary_csv}")

# Create zip archive for one-click download
zip_path = Path(BASE_DIR) / "gate_output.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUT_DIR):
        for file in files:
            f_path = Path(root) / file
            zipf.write(f_path, arcname=f_path.relative_to(OUT_DIR))
print("\n" + "=" * 75)
print(f"ALL RESULTS PACKAGED! Download: {zip_path}")
print("=" * 75)